On DAP, we can't install ipykernel in `.venv`. Thus, we need to apply a different strategy.

* do `unzip -o graph-ensemble.zip` 
If not working, remove `mc rm -rf graph-ensemble` and, then, `unzip graph-ensembles.zip`;
* `deactivate` (the `virtual-env`) such that the standard jupter kernel is selected;
* install via `pip install --editable . --config-settings editable_mode=compat`
* create a cell down here and do `import graph_ensembles` to check for the installation;

Usefull Commands:
* For this project, ```vars``` and ```plots``` are saved in `./outputs`;
* Delete folders on dap: ```mc rm --force --purge folder```
* To save new plots, delete previous ones with
```mc rm --force --purge dap/corealgos/rmilocco/outputs/datasets/ING-Directed```

On local:
* Linux: Zip outside the ``graph-ensembles`` folder: 
<br>```zip -rX graph-ensembles.zip graph-ensembles -x "graph-ensembles/src/graph_ensembles.egg-info/*" ".*" "*/.*" "*/__pycache__/*" "*/sythetic_network/*"```

Run the right install cmd from [pytorch](https://pytorch.org/), based on your architecture

Claim: by reconstructing the unobserved, we may close the gap between the total Page-Rank and Page-Rank only inside the ING-clients

1) Split the nodes into ING (`vI`) and `ROW` (`vR`);

2) Find ``vI`` and ``eI`` as the edges only between. We will cal `intra` (`eI`) edges, `bet` (ING-ROW), `row` (non ING interacting clients); 

4) Calculate the Page-Rank of only the `intra` nodes and compare it with the full graph PR.
Now, we expect that the 2 PR are different. So, help this bias by reconstructing the missing part. Ref [LateX](https://asajadi.github.io/fast-pagerank/) based on [MathWorks](https://www.mathworks.com/content/dam/mathworks/mathworks-dot-com/moler/exm/chapters/pagerank.pdf);

5) Calculate the strengths taking into account also the ING-ROW fluxes, while discarding the self-payments;
6) Freeze the `eI` and fit $\delta$ parameter as 
    * $L_I \stackrel{!}{=} \langle L_I \rangle(\delta_I) := \sum_{i \in I, j \in I} p_{ij}(\delta_I)$;

    * $L^{no-I}_U = L_I + L_{bet} \stackrel{!}{=} L_I + \sum_{i \in I, r \in R} (p_{ir}(\delta_{U}) + p_{ri}(\delta_{U})) $;

    * $L_U = L_I + L_{bet} \stackrel{!}{=} \sum_{(i,j) \in \left\{(I,I),(I,R),(R,I) \right\} } p_{ij}(\delta_{U}) $;

7) Page-Rank (or Influence Vector);

In [1]:
# auto-reload the packages at every run
%load_ext autoreload
%autoreload 2

#display all the results not only the last one
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import os
try:
    corpkey = True if os.environ['DSBOX_USERNAME'] else None
    # %pip install matplotlib pandas scipy tqdm torch torchvision
except:
    corpkey = None

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import graph_ensembles as ge
from graph_ensembles import sparse as sp
import graph_ensembles.utils as utils
import graph_ensembles.dependencies as dep
from graph_ensembles.plots import plotting_functions as plot

# plot.plot_local_fonts(corpkey)
utils._set_mpl_params(fontsize=20)
utils.check_cpu_gpu_with_torch()

dataset_name = "ING" #"recNET"
dataset_direction = "Directed"
id_code, cg_method, year = "grid_id", "random", 2022


-Logical CPUs: 22
-GPUs in use:
-Number of GPUs: 1
  GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU
    - Compute Capability: 89
    - Total Memory: 8187 MB
    - Multiprocessors: 24
-Current CUDA device: 0
-CUDA version: 12.8
-total_cores: 3072


In [2]:
# note that inside the kwargs there is a copy of the pdtrans
max_num_entries = None #if corpkey else 3e3 #30e6 # old 1e3
pdtrans, kwargs, total_levels = \
    ge.dataset_loader(dataset_name, dataset_direction = dataset_direction,
                    corpkey = corpkey, id_code = id_code, cg_method = cg_method,
                    year = year, max_num_entries = max_num_entries) #63332573

pdtrans.info()


Reading from local source
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3938370 entries, 0 to 3938369
Data columns (total 5 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   payer_grid_id           int64  
 1   payer_naics_code        int64  
 2   beneficiary_grid_id     int64  
 3   beneficiary_naics_code  int64  
 4   amount_euro             float64
dtypes: float64(1), int64(4)
memory usage: 150.2 MB


Define all the vertex and edges

In [3]:
e = pdtrans.loc[:, [f'payer_{id_code}', f'beneficiary_{id_code}']]
unique_nodes_from = lambda df: pd.DataFrame(data = np.unique(df.to_numpy().ravel('K')), columns = ["id"])
v = unique_nodes_from(e) # = 1 column DataFrame with "id" the node label, whereas the index is the incresing integer node-index
e = pdtrans.iloc[:, ::2] # payer, beneficiary, amount_euro
e.columns = ["src", "dst", "amount"] 

e.head()
print(f'-v.shape: {v.shape}',)

,src,dst,amount
0,0,292,2257.270901
1,0,1353,1010.457165
2,0,3118,22307.329849
3,0,3230,1998.664794
4,0,4559,485.497533


-v.shape: (66146, 1)


Create Full Graph

In [4]:
ivec_name = "page_rank"
kwargs_graph = {'name' : 'ING', 'level' : 0, 'corpkey' : corpkey, "_pr_name" : ivec_name}
g = sp.graphs.DiGraph(v, e, **kwargs_graph)
# g.load_or_create_degrees()

del pdtrans

In [5]:
g._kwargs_pr = {"p" : 0.85, "max_iter" : 200, "tol" : 1e-06}
g._pr = g.pagerank_power(**g._kwargs_pr)
# g.save_vars()

Create a split of ``v`` into `vI`

Fit the `delta_intra`, for a splitting

In [6]:
# NB: the vars are not saved since I commented save_vars, savetxt and makedirs

In [ ]:
from os.path import dirname as up
from tqdm import trange
from itertools import product

vsplits, intra_sizes = [0], sorted([1])
x0 = [1.8996372e-17]

# sample arguments
num_graph_samples_per_vsplit = 11
chunk_row_size = 500
measures = ["_pr", "_out_degree", "_in_degree", "_annd_out_out", "_annd_in_in",]
rtol = 1e-8
fit_methods = ["num_edges_intra"] #["num_edges_" + i for i in ["intra", "intra_bet", "bet"]]

# for intra_size in intra_sizes:
#     for vsplit in vsplits:
for intra_size_vsplit in product(intra_sizes, vsplits):
    intra_size, vsplit = intra_size_vsplit

    vI, eI, idx_intra_nodes = g.vsplit_intra(v, e, intra_size=intra_size, vsplit=vsplit)


    fit_methods = ["num_edges_intra"] if intra_size == 1 else fit_methods # since intra_sizes are sorted, this line changes only the last it

    for fit_method in fit_methods:
        
        vR, num_edges_bet = g.vsplit_row(v, vI, e, idx_intra_nodes, fit_method)
        
        kwargs_model = kwargs_graph.copy()
        kwargs_model.update({"name" : "MultiScaleMod", "fit_method" : fit_method})
        
        if intra_size == 1:
            unsampled_vI = np.zeros(g.num_vertices, dtype=np.bool_)
            kwargs_model.update({"prop_out_I" : g.out_strength(), "prop_in_I" : g.in_strength(),
                                "prop_out_R" : np.array([0]), "prop_in_R" : np.array([0]), 
                                "intra_size" : 1})
            model = sp.ScaleInvariantModel(g, **kwargs_model)

            # load or fit the param
            gI = g


        else:
            # vsplit nodes, edges in interal (vsplit = seed)
            vI, eI, vR, num_edges_bet = g.vsplit_intra_row(v, e, intra_size=intra_size, vsplit=vsplit, fit_method=fit_method)

            # mask unsampled_vI for fast sampling, frozen edges (integer eI)
            unsampled_vI, frozen_edges, gI = g._set_frozen_edges_gI(intra_size, vsplit, vI, eI, kwargs_graph)
            
            # compute the page-rank only in the internal part
            gI._pr = gI.pagerank_power(**g._kwargs_pr)    
            
            # 1) METTI TUTTO DENTRO UNA FUNZIONE PER CREARE GLI ARGOMENTI MINIMI CHE TI SERVONO;
            # 2) exp_edges non calcolano il num dei link nella stessa maniera in cui fitti
            vR_nodes = vR.T.values[0]
            out_stre = lambda i: g.out_strength()[g.id_dict[i]]
            in_stre = lambda i: g.in_strength()[g.id_dict[i]]
            cmap = lambda stre, nodes: np.array(list(map(stre, nodes)))
            kwargs_model.update({
                            "prop_out_R" : cmap(out_stre, vR_nodes), "prop_in_R" : cmap(in_stre, vR_nodes), 
                            "prop_out_I" : gI.out_strength(), "prop_in_I" : gI.in_strength(),
                            "num_vertices" : len(v) if "bet" in fit_method else len(vI),
                            "num_edges" : sp.ScaleInvariantModel.num_edges_fit(gI.num_edges(), num_edges_bet, fit_method),
                            "level" : g.level, "intra_size" : gI.intra_size, "vsplit" : gI.vsplit,
                            })
            model = sp.ScaleInvariantModel(**kwargs_model)

        model.load_or_fit(x0 = x0[0], maxiter = 30, verbose = 2)

        # if intra_size < 1, set the right strengths to predict the full-network
        if intra_size < 1: model.set_num_vertices_out_in_strengths_to(g)
        
        # create vars dir
        num_sampled_graphs = model.set_ensemble_variables(measures, num_graph_samples_per_vsplit)

        print(f'\n-For vsplitting vsplit {vsplit}: {num_sampled_graphs} graphs already sampled, {np.clip(num_graph_samples_per_vsplit - num_sampled_graphs, 0, None)} remaining')

        # send variables (model.params, ...) to cuda:0
        unsampled_vI = model.send_variables_to_gpu(unsampled_vI)

        # track max of ens std for early stopping
        max_ivec_std = []

        # set the page rank on g, gI
        utils.set_ivec_on_I(g, gI)
        gI.calculate_measures(g, measures)
        g.calculate_measures(g, measures)

        # def var for model vars
        mod_vars = model.__dict__
        
        # Note: the seed = vsplit only works for solo-agent. The graph-sampling is parallelized, so it would be random even if vsplit specified.
        # That's why .sample() has graph_idx as argument
        # set graph_idx = 0, since if all the graphs are already sampled in the next calculations it will have a number different to None
        for graph_idx in trange(num_sampled_graphs, num_graph_samples_per_vsplit, 
                        desc=f"-Total progress {int(np.round(vsplit/len(vsplits) * 100))}%, Inner Graph Sampling", position = 0, leave= True):
            
            # only at 1st iteration
            if graph_idx == num_sampled_graphs:
                for m in measures:
                    mod_vars[f"prev{m}"] = mod_vars[m].copy()
                    mod_vars[f"prev{m}_std"] = mod_vars[m+"_std"].copy()

            # sample
            gs = model.sample(ref_g = g, unsampled_vI = unsampled_vI, frozen_edges = frozen_edges, 
                            graph_idx = graph_idx, chunk_row_size = chunk_row_size)
            gs.calculate_measures(g, measures)
            # gs.save_vars(name = f"graph{gs.graph_idx}")
            
            for m in measures:
                mod_vars[m], mod_vars[m+"_std"] = \
                    model.recursive_mean_std(graph_idx, mod_vars[f"prev{m}"], mod_vars[f"prev{m}_std"], gs.__dict__[m])
            
            # track only the changing in ivec
            max_ivec_std.append(np.max(model._pr_std))

            # create 10 snapshot of the sampling vars and plots
            num_sampled_graphs = graph_idx + 1
            # step = num_graph_samples_per_vsplit // 10
            if num_sampled_graphs == num_graph_samples_per_vsplit: #num_sampled_graphs % step == 0 and num_sampled_graphs > 1:
                
                # save the mean and std as [[mean],[std]]
                # for m in measures:
                    # fname = model.vars_dir_ensembles + f"/{m}/num_samples_{num_sampled_graphs}.csv"
                    # np.savetxt(fname, X = np.vstack((mod_vars[m], mod_vars[f"{m}_std"])))

                # generate the plot of page-ranks
                num_bins = int(np.sqrt(model.num_vertices))
                utils.set_model_ivec_on_I(model, gI)

                # rescale the internal page-rank only in the last step
                if num_sampled_graphs == num_graph_samples_per_vsplit:
                    gI.rescale_ivec_with(model)

                plot.ivec_on_internal_nodes(model, g, gI, num_bins)
                plot.ivec_on_internal_nodes_vs_rank(model, g, gI, num_sigmas = 2)
                plot.topN_overlap_rel_err(g, gI, model)
                plot.norm_diffs_per_iteration(model.plots_dir, max_ivec_std)
                
            # if the max variance is low, stop sampling
            if np.max(model._pr_std) < rtol and graph_idx > 0:
                print(f'-Breaking since max of ens std {np.round(np.max(model._pr_std), 5)} < {rtol}',)
                break
            
            # redefine prev ivec and std
            for m in measures:
                mod_vars[f"prev{m}"] = mod_vars[m].copy()
                mod_vars[f"prev{m}_std"] = mod_vars[m+"_std"].copy()
        
        # compute the expected degree and plt them
        unsampled_vI = model.send_variables_to_cpu(unsampled_vI)
        _ = model.expected_degree(unsampled_vI, gI)
        plot.ccdf_deg_out_in(g, gI, model)
        plot.exp_deg_out_in(g, gI, model)
        plot.annd_vs_deg(g, gI, model, measures)

        # model.delete_ensemble_samples(True)

-Fit the parameter with num_edges_intra
x0 = [1.8996372e-17]
|f(x0)| = 3882300.684314696
    Iteration 1
    fun = -2355945.6081348015
    fun_prime = 6.682879727814574e+20
    dx = 1.5628290190171345e-15
    x = [1.58182539e-15]
    |f(x)| = 2355945.6081348015
    diff = [82.26986811]
 
    Iteration 2
    fun = -567047.6487111957
    fun_prime = 4.110866913318451e+20
    dx = 3.525344917295466e-15
    x = [5.10717031e-15]
    |f(x)| = 567047.6487111957
    diff = [2.22865617]
 
    Iteration 3
    fun = -30060.565581535455
    fun_prime = 3.697259263573383e+20
    dx = 1.3793870263084065e-15
    x = [6.48655733e-15]
    |f(x)| = 30060.565581535455
    diff = [0.27008832]
 
    Iteration 4
    fun = -84.00553032848984
    fun_prime = 3.676653038061173e+20
    dx = 8.130499767138879e-17
    x = [6.56786233e-15]
    |f(x)| = 84.00553032848984
    diff = [0.01253438]
 
    Iteration 5
    fun = -0.0006559710018336773
    fun_prime = 3.676595618928588e+20
    dx = 2.2848370368064125e-19
 

-Total progress 0%, Inner Graph Sampling: 100%|██████████| 11/11 [00:48<00:00,  4.37s/it]


In [34]:
self = model
self.exp_edges(
            self.num_edges_i,
            self.param,
            self.prop_out_I,
            self.prop_in_I,
            self.prop_out_R,
            self.prop_in_R,
            self.prop_dyad,
            self.selfloops,
            self.fit_method,
        )

3938370.0051127826

In [36]:
g.num_edges()

np.int64(3938370)

In [ ]:
def overlap_at_k(g, gI, gs):

    N = g._pr_on_I.size 
    
    start, stop, step = 1, N, 25
    if N > 100:
        intervals = np.geomspace(start, stop, step, dtype=int)
    else: 
        intervals = [1] + list(range(step, stop + 1, step)) #[1] + [step_top_N*i for i in range(1, num_points+1)]

    topN_arr = lambda v: [v[:i] for i in intervals]

    # observed (this should be done outside the sampling loop)
    g_topN_rank = topN_arr(g._pr_rank_on_I)

    # expected
    gs_topN_rank = topN_arr(gs._pr_rank_on_I)

    # 1. Calculate overlap between g_topN_rank and model_topN_rank
    overlap_perc = lambda r: [np.intersect1d(g_topN, exp_topN).size / g_topN.size for g_topN, exp_topN in zip(g_topN_rank, r)]
    g_gs_overlap = overlap_perc(gs_topN_rank)

    # 2. Calculate the total page-rank error
    g_topN_ivec = topN_arr(g._pr_on_I)
    gs_topN_ivec = topN_arr(gs._pr_on_I)

    topN_rel_err = lambda r: [utils.rel_err_norm(exp_topN, g_topN) * 100 for g_topN, exp_topN in zip(g_topN_ivec, r)]

    g_gs_rel_err = topN_rel_err(gs_topN_ivec) 

In [22]:
from itertools import product

vsplits, intra_sizes = [0], [0.2, 0.6]
list(product(intra_sizes, vsplits))

[(0.2, 0), (0.6, 0)]

In [ ]:

vR_nodes = vR.T.values[0]
out_stre = lambda i: g.out_strength()[g.id_dict[i]]
in_stre = lambda i: g.in_strength()[g.id_dict[i]]
cmap = lambda stre, nodes: np.array(list(map(stre, nodes)))
kwargs_model.update({
                "prop_out_R" : cmap(out_stre, vR_nodes), "prop_in_R" : cmap(in_stre, vR_nodes), 
                "prop_out_I" : gI.out_strength(), "prop_in_I" : gI.in_strength(),
                "num_vertices" : len(v), "num_vertices_int" : gI.num_vertices,
                "level" : g.level, "intra_size" : gI.intra_size, "vsplit" : gI.vsplit,
                "num_edges" : sp.ScaleInvariantModel.num_edges_fit(gI.num_edges(), num_edges_bet, fit_method),
                })

model = sp.ScaleInvariantModel(**kwargs_model)

In [26]:
model.fit(
        x0=x0,
        method="num_edges",
        )

In [27]:
model.param

array([[3.06203729e-09]])

In [ ]:
from numba import prange, njit

@njit()
def num_edges_jac_i(fun, jac, d, x_i, y_j, z_ij = 1.0):
    """Compute the probability of connection and the jacobian
    contribution of node i and j.
    """
    tmp = x_i * y_j * z_ij
    tmp1 = d[0] * tmp
    
    # vars are immutable so return the updated value
    # print(f'-p_iI: {- np.expm1(-tmp1)}',)
    return fun - np.expm1(-tmp1).sum(), jac + (tmp * np.exp(-tmp1)).sum()
    
def num_edges_fit_fun(self, delta):

    N = len(self.prop_out_I)

    # Preallocate result vectors for each outer loop iteration (i)
    # These arrays store intermediate totals per i, which can be summed later
    f_vector = np.zeros(N)
    jac_vector = np.zeros(N)

    # Outer loop is parallelized with prange
    # This is the correct and efficient use of numba's parallelism
    for i in prange(N):
        
        # Use scalar accumulators for better memory efficiency and cache usage
        f_i = 0.0
        jac_i = 0.0
        
        prop_out_i, prop_in_i = self.prop_out_I[i], self.prop_in_I[i]
        
        if "intra" in self.fit_method:
            # print(f'\n-i: {i}',)
            f_i, jac_i = num_edges_jac_i(f_i, jac_i, delta, prop_out_i, self.prop_in_I)
            # print(f'-f_i: {f_i}',)
            f_i, jac_i = num_edges_jac_i(f_i, jac_i, delta, self.prop_out_I, prop_in_i)
            f_i /= 2
            jac_i /= 2
            # print(f'-f_i: {f_i}',)

        if "bet" in self.fit_method:
            # print(f'-bet',)
            f_i, jac_i = num_edges_jac_i(f_i, jac_i, delta, prop_out_i, self.prop_out_R)
            f_i, jac_i = num_edges_jac_i(f_i, jac_i, delta, self.prop_out_R, prop_in_i)

        # Store per-node results
        f_vector[i] = f_i
        jac_vector[i] = jac_i
        # print(f'-f_vector: {f_vector}',)

    # Sum across all nodes to get final result (parallel reduction is fast for large n)
    f_vector, jac_vector = np.sum(f_vector), np.sum(jac_vector)

    if "intra" in self.fit_method:
        # discard self-loops
        # print(f'-\n SelfLoops',)
        f_vector, jac_vector = num_edges_jac_i(-f_vector, -jac_vector, self.param, self.prop_out_I, self.prop_in_I)
        
        f_vector *= -1
        jac_vector *= -1

    return f_vector, jac_vector

num_edges_fit_fun(model) #self.param, self.prop_out_I[0], self.prop_in_I)

Comparison of new code to calculate the function and jacobian

In [ ]:
omodel = sp.ScaleInvariantModel(gI, **kwargs_model)
omodel.param = [0.3]
omodel.num_edges_fit_fun(omodel.param)

omodel = sp.ScaleInvariantModel(gI, **kwargs_model)
# omodel.prop_out = np.arange(3)
# omodel.prop_in = np.arange(3)
omodel.param = [0.3]
omod_ne, omod_jac = omodel.num_edges_fit_fun(omodel.param)
omod_ne, omod_jac

frmv_diag = lambda A: A * (1 - np.eye(A.shape[0]))

prop_out, prop_in = omodel.prop_out[:, None], omodel.prop_in[None, :]
arg = omodel.param * prop_out @ prop_in
P = frmv_diag(-np.expm1(-arg))
# print(f'-P:\n {P}',)
P.sum()

(2413.162882249104, 46.362625047975285)

np.float64(2413.1628822491048)

## Miscellanea

### Check if num edges with frozen edges is correct

Compare matrix calculation (numpy) with discarding frozen edges in the for-loop (numba)

In [ ]:
frmv_diag = lambda A: A * (1 - np.eye(A.shape[0]))

# def the argument of the MSM p_ij
arg = model.param * model.prop_out[:, None] @ model.prop_in[None, :]

# compute the p_ij
p_ij = -np.expm1(-arg)
print(f'-p_ij.shape: {p_ij.shape}',)

# mask not to sample the internal nodes
unsampled_ij = ~(unsampled_vI[:, None] @ unsampled_vI[None, :]).astype(bool)
print(f'-unsampled_ij.shape: {unsampled_ij.shape}',)

# filter the off-diagonal and not-frozen edges
unsampled_pij = frmv_diag(p_ij)[unsampled_ij]

# find their sum, i.e. off-diagonal and not-frozen number of edges
num_edges_matrix = unsampled_pij.sum()
print(f'-num_edges_matrix: {num_edges_matrix}',)

# check if its the same with the one computed internally
num_edges_matrix += gI._num_edges
model.expected_num_edges(recompute = True, unsampled_vI = unsampled_vI, num_frozen_edges = gI._num_edges)

utils.signed_rel_err(num_edges_matrix, model._exp_num_edges)

### Compute observed Page-Ranks on the splitted networks

In [ ]:
# from os.path import dirname as up

# gI_dirfunc = lambda intra_size, vsplit: up(up(g.vars_dir)) + f"/intra_size{intra_size}/vsplit{vsplit}/level{g.level}"

# for intra_size in intra_sizes:
# 	for vsplit in range(num_vsplits):
        
# 		# vsplit nodes, edges in interal
# 		if not os.path.exists(gI_dirfunc(intra_size, vsplit)):
# 			vI, eI = g.vsplit_intra_row(v, e, intra_size=intra_size, vsplit=vsplit)

# 			# update the graph name
# 			kwargs_graph.update({'graph_kind': "intra", "intra_size" : intra_size, "vsplit" : vsplit})
# 			gI = sp.graphs.DiGraph(vI, eI, **kwargs_graph)

# 			# compute the page-rank only in the internal part
# 			gI.page_rank = gI.pagerank_power(**g._kwargs_pr)

# 			gI.save_vars()
            
# plot.pagerank_on_internal_nodes(g, gI_dirfunc, intra_size, num_vsplits,)

### Sampling with GPUs in Torch or CPUs with numba

Check if `unsampled_vI` entries are fixed

In [ ]:
import numpy as np
from itertools import product

# select the unsampled_vI
uns_id = np.where(unsampled_vI)[0]

# create a mapping from node to id
matidx_2_id = {k:v for k,v in enumerate(uns_id)}

# create a meshgrid
rows, cols = np.array(list(product(uns_id, repeat=2))).T
uns_N = len(uns_id)
print(f'-uns_N: {uns_N}',)

# filter the gs.adj for all the uns_id pairs
mat = gs.adj[rows, cols].reshape(uns_N, uns_N)

# find where mat is true
mat_rows, mat_cols = np.where(mat)

# create the DataFrame mapped into full-id nodes
frozen_edges_mat = pd.DataFrame({"_src" : mat_rows, "_dst" : mat_cols}).map(matidx_2_id.get)

# check the edges are the same
frozen_edges_mat.eq(frozen_edges).all()

-uns_N: 15


np.True_

Do Page-Rank for `gU`

In [ ]:
# find idx of edges containing v and filter edges
idx_at_least_one_ = lambda vI: e['src'].isin(vI['id']) | e['dst'].isin(vI['id'])

# containing vI, vR
idx_eU = idx_at_least_one_(vI)

# select the edge ING
eU = 3
# eU = edge_idx(idx_eU)

# reselect vU, since using v leads to inactive nodes not well-handled by the DiGraph
vU = unique_nodes_from(eU.iloc[:,:2])

In [ ]:
# kwargs_graph.update({'intra_size' : intra_size, 'vsplit' : vsplit, 'graph_kind' : "intra_inter",})
gU = sp.graphs.DiGraph(vU, eU, **kwargs_graph)
# g.load_or_create_degrees()

Compare the United PageRank vs Full PageRank over all the `vU` 

In [ ]:
pr_gU = pagerank_power(gU.adj, p=0.85, max_iter=100,
                   tol=1e-06, personalize=None, reverse=True)

In [ ]:
idx_IntraNode2United = list(map(lambda x: gU.id_dict.get(x), gI.id_dict))
pr_gU_on_Intra = pr_gU[idx_IntraNode2United]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([pr_g_on_I.min(), pr_g_on_I.max()],
        [pr_g_on_I.min(), pr_g_on_I.max()],
        'r--')

ms, alpha = 30, 0.6
ax.scatter(pr_g_on_I, pr_gI, alpha=alpha, label = f"Intra PR", marker = 'o', c = dep.obs_color, s = ms)
ax.scatter(pr_g_on_I, pr_gU_on_Intra, alpha=alpha, label = f"United PR", marker = 'x', c = dep.ref_model_color, s = ms)
ax.set(
    xlabel='Full-PR on Intra',
    ylabel='Intra/United',
    title='PageRank: Intra/United VS Full',
    xscale='log',
    yscale='log'
)
leg = ax.legend()
for lh in leg.legend_handles:
    lh.set_alpha(1)
    lh.set_sizes([60])
ax.grid(False)

utils.save_fig(fig, full_path = g.plots_base_dir + "/PageRanks_on_Intra.png")
plt.close()

[Text(0.5, 0, 'Full-PR on Intra'),
 Text(0, 0.5, 'Intra/United'),
 Text(0.5, 1.0, 'PageRank: Intra/United VS Full'),
 None,
 None]